In [ ]:
import urllib, urllib.request, time
import arxiv
import pandas as pd
import urllib
import requests
import json
import csv
from collections import Counter, defaultdict
import numpy as np 
import matplotlib.pyplot as plt 
%matplotlib inline 
import datetime


def fetch_arxiv(query='all:electron', start=0, max_results=1, retries=5):
    url = f'https://export.arxiv.org/api/query?search_query={query}&start={start}&max_results={max_results}'
    headers = {'User-Agent': 'MyApp/1.0 (your@email.com)'}
    
    for attempt in range(retries):
        wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
        print(f"Attempt {attempt + 1} — waiting {wait}s...")
        time.sleep(wait)
        
        try:
            req = urllib.request.Request(url, headers=headers)
            data = urllib.request.urlopen(req)
            return data.read().decode('utf-8')
        except urllib.error.HTTPError as e:
            if e.code == 429:
                print(f"Rate limited (429). Retrying...")
            else:
                raise
    
    raise Exception("Max retries reached. Try again later.")

result = fetch_arxiv()
print(result)


results_generator = arxiv.Client(
  page_size=1000,
  delay_seconds=3,
  num_retries=3
).results(arxiv.Search(
  query='"quantum dots"',
  id_list=[],
  sort_by=arxiv.SortCriterion.Relevance,
  sort_order=arxiv.SortOrder.Descending,
))

quantum_dots = []
for paper in results_generator:
  quantum_dots.append(paper)



qd_df = pd.DataFrame([vars(paper) for paper in quantum_dots])


qd_df['first_author'] = [authors_list[0].name for authors_list in qd_df['authors']]
qd_df['_result'] = quantum_dots


qd_df = qd_df[['title', 'published', 'first_author', '_result']]
qd_df
QUERY = """
(
all:chatbot OR
all:"artificial intelligence" OR
all:"generative AI" OR
all:GenAI OR
all:"large language model" OR
all:LLM OR
all:LLMs
)
AND
(
all:loneliness OR
all:"social isolation" OR
all:"social alienation" OR
all:"social exclusion" OR
all:"social withdrawal" OR
all:"social connection" OR
all:"social inclusion" OR
all:belonging OR
all:"social satisfaction" OR
all:"perceived cohesion" OR
all:"social well-being" OR
all:"social wellbeing" OR
all:"interpersonal well-being" OR
all:"interpersonal wellbeing"
)
"""

client = arxiv.Client(
    page_size=100,
    delay_seconds=5,
    num_retries=10
)

search = arxiv.Search(
    query=QUERY,
    max_results=10000,
    sort_by=arxiv.SortCriterion.Relevance
)

papers = []

for paper in client.results(search):

    papers.append({
        "title": paper.title,
        "authors": "; ".join(a.name for a in paper.authors),
        "published": paper.published,
        "updated": paper.updated,
        "summary": paper.summary.replace("\n", " "),
        "categories": "; ".join(paper.categories),
        "pdf_url": paper.pdf_url,
        "entry_id": paper.entry_id,
    })

df = pd.DataFrame(papers)

print(f"Found {len(df)} papers")

df.to_csv(
    "arxiv.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved to arxiv.csv")

: 